# Task 1 — Netflix Data Cleaning & Preparation

Prepare and organize the Netflix dataset for business analytics and reporting.

**Project:** Auspify Technologies — Data Analysis Using Python Internship  
**Dataset:** Netflix catalogue CSV supplied for this internship  
**Tools:** Python, Pandas, NumPy

#### Importing Libraries

In [1]:
import pandas as pd
import numpy as np

## 1. Load the dataset

In [2]:
df = pd.read_csv("Dataset (1).csv")
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
df.head()

Rows: 8,790 | Columns: 10


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,2021-09-25,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,2021-09-24,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,2021-09-22,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,2021-09-24,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


## 2. Profile data quality
Review column names, data types, actual nulls, duplicate rows, and placeholder text. `Not Given` is text, so Pandas does not count it as a null until we replace it.

In [3]:
print("Columns:", df.columns.tolist())
print("\nData types and non-null counts:")
df.info()
print("\nActual null values:")
display(df.isna().sum().to_frame("missing_count"))
print("\nFully duplicated rows:", int(df.duplicated().sum()))

for col in df.select_dtypes(include="object").columns:
    n = df[col].astype(str).str.strip().str.lower().eq("not given").sum()
    if n:
        print(f"{col}: {n:,} 'Not Given' placeholders")

Columns: ['show_id', 'type', 'title', 'director', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in']

Data types and non-null counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8790 entries, 0 to 8789
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8790 non-null   object
 1   type          8790 non-null   object
 2   title         8790 non-null   object
 3   director      8790 non-null   object
 4   country       8790 non-null   object
 5   date_added    8790 non-null   object
 6   release_year  8790 non-null   int64 
 7   rating        8790 non-null   object
 8   duration      8790 non-null   object
 9   listed_in     8790 non-null   object
dtypes: int64(1), object(9)
memory usage: 686.8+ KB

Actual null values:


,missing_count
show_id,0
type,0
title,0
director,0
country,0
date_added,0
release_year,0
rating,0
duration,0
listed_in,0



Fully duplicated rows: 0
director: 2,588 'Not Given' placeholders
country: 287 'Not Given' placeholders


## 3. Standardize column names and text
Column names are converted to lowercase. Leading/trailing spaces are removed from text values.

In [4]:
df.columns = (
    df.columns.str.strip().str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

print(df.columns.tolist())

['show_id', 'type', 'title', 'director', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in']


## 4. Convert placeholders and clean missing values
Convert `Not Given` to real missing values first. For categorical fields, use `Unknown` so that records remain available for analysis. Dates that cannot be parsed remain missing rather than being invented.

In [5]:
df.replace(r"(?i)^not given$", np.nan, regex=True, inplace=True)

for col in ["director", "country", "rating", "duration"]:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown")

df["date_added"] = pd.to_datetime(df["date_added"], errors="coerce")
df["release_year"] = pd.to_numeric(df["release_year"], errors="coerce").astype("Int64")

print("Missing values after handling:")
display(df.isna().sum().to_frame("missing_count"))

Missing values after handling:


,missing_count
show_id,0
type,0
title,0
director,0
country,0
date_added,0
release_year,0
rating,0
duration,0
listed_in,0


## 5. Standardize key categorical fields
Standardize content type and trim country/rating text. Country values remain comma-separated here; Task 3 splits them into individual country records for counting.

In [6]:
df["type"] = df["type"].astype("string").str.strip().str.title()

df["country"] = df["country"].apply(
    lambda x: ", ".join(part.strip() for part in x.split(","))
    if isinstance(x, str) and x != "Unknown" else x
)

df["rating"] = df["rating"].apply(
    lambda x: x.strip().upper()
    if isinstance(x, str) and x != "Unknown" else x
)

print("Content types:")
display(df["type"].value_counts(dropna=False).to_frame("count"))

Content types:


,count
type,
Movie,6126
Tv Show,2664


## 6. Check duplicates and identifiers
Remove exact duplicate rows. Check duplicate IDs separately; this notebook keeps the first row per `show_id` if duplicates exist, and reports how many were removed.

In [7]:
before_exact = len(df)
df = df.drop_duplicates()
exact_removed = before_exact - len(df)

duplicate_ids = int(df["show_id"].duplicated().sum())
before_id_dedup = len(df)
df = df.drop_duplicates(subset=["show_id"], keep="first")
id_rows_removed = before_id_dedup - len(df)

print("Exact duplicate rows removed:", exact_removed)
print("Rows removed for duplicate show_id:", id_rows_removed)
print("Final shape:", df.shape)

Exact duplicate rows removed: 0
Rows removed for duplicate show_id: 0
Final shape: (8790, 10)


## 7. Validate and export
Export the cleaned dataset. Tasks 2–6 use this file as their shared input.

In [8]:
print("Final shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum().to_frame("missing_count"))

print("\nDuplicates:", df.duplicated().sum())
print("\nData types:")
print(df.dtypes)

df.to_csv("netflix_cleaned.csv", index=False)

check = pd.read_csv("netflix_cleaned.csv")
print("Reloaded shape:", check.shape)
display(check.head())
print("Cleaned dataset saved successfully.")

Final shape: (8790, 10)

Missing values:
              missing_count
show_id                   0
type                      0
title                     0
director                  0
country                   0
date_added                0
release_year              0
rating                    0
duration                  0
listed_in                 0

Duplicates: 0

Data types:
show_id                 object
type            string[python]
title                   object
director                object
country                 object
date_added      datetime64[ns]
release_year             Int64
rating                  object
duration                object
listed_in               object
dtype: object
Reloaded shape: (8790, 10)


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,2021-09-25,2020,PG-13,90 min,Documentaries
1,s3,Tv Show,Ganglands,Julien Leclercq,France,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,Tv Show,Midnight Mass,Mike Flanagan,United States,2021-09-24,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,2021-09-22,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,2021-09-24,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


Cleaned dataset saved successfully.
